# 09 | Initial Supervised Regression Candidates
**NHS A&E Intelligent Decision Support System**

**Objective:** demonstrate reproducible, leakage-safe fitting of our three initial supervised regression candidates using the exact 13 core predictors and the January 2025 publication-aware historical training fold (`decision_date = 2024-12-31`, `target_period <= 2024-11`):
1. **Ridge Regression** (`StandardScaler` + `Ridge` in a scikit-learn `Pipeline`)
2. **Random Forest Regressor** (`RandomForestRegressor`, `random_state=42`)
3. **Histogram Gradient Boosting** (`HistGradientBoostingRegressor`, `early_stopping=False`, `random_state=42`)

**Important:** This notebook verifies model construction, training-only preprocessing, and prediction generation only. Any predictions shown below are strictly **in-sample training-fold diagnostics**, not out-of-sample generalisation evidence. Neither the 2025 validation set nor the 2026 final test set is evaluated here (reserved for Step 9+).

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/model_input_manifest.json").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the project repository before running this notebook.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.evaluation_splits import assign_chronological_splits, load_verified_features
from src.models.feature_engineering import TARGET_COLUMN
from src.models.supervised_models import (
    INITIAL_DEMO_TARGET_MONTH,
    MODEL_NAMES,
    extract_fitted_metadata,
    fit_model,
    get_core_feature_columns,
    get_environment_versions,
    predict_with_model,
    select_training_fold,
)

feat_df, provenance = load_verified_features(project_root)
split_df = assign_chronological_splits(feat_df)
train_df, fold_info = select_training_fold(split_df, INITIAL_DEMO_TARGET_MONTH, complete_core_only=True)
core_cols = get_core_feature_columns()

print("Environment:", get_environment_versions())
print("M06 SHA-256:", provenance["m06_csv_sha256"])
print("Feature CSV SHA-256:", provenance["feature_csv_sha256"])
display(pd.Series(fold_info, name="January 2025 Initial Training Fold").to_frame())


## 1. Approved 13 Core Predictors, Completeness, and Target Distribution

In [ ]:
feature_audit = pd.DataFrame({
    "feature": core_cols,
    "missing_in_train_fold": [int(train_df[c].isna().sum()) for c in core_cols],
    "mean": [round(float(train_df[c].mean()), 3) for c in core_cols],
    "std": [round(float(train_df[c].std()), 3) for c in core_cols],
    "min": [round(float(train_df[c].min()), 3) for c in core_cols],
    "max": [round(float(train_df[c].max()), 3) for c in core_cols],
})
display(feature_audit)
display(train_df[TARGET_COLUMN].astype(float).describe().round(1).to_frame(name="Training Target Distribution (2023-03..2024-11)"))


## 2. Instantiate and Fit All Three Regression Candidates
Each pipeline fits strictly on the identical `2,567` complete-core historical training rows (`target_period <= 2024-11`).

In [ ]:
fitted = {}
summary_rows = []
diag_df = train_df[["org_code", "origin_period", "decision_date", "target_period", TARGET_COLUMN]].copy()

for name in MODEL_NAMES:
    pipe = fit_model(name, train_df, feature_columns=core_cols)
    preds = predict_with_model(pipe, train_df, clip_non_negative=True, feature_columns=core_cols)
    meta = extract_fitted_metadata(name, pipe)
    fitted[name] = pipe
    diag_df[f"in_sample_pred_{name}"] = preds["prediction"].round(1).to_numpy()
    summary_rows.append({
        "model": name,
        "estimator": meta["estimator_class"].split(".")[-1],
        "preprocessing": meta["preprocessing"]["step"],
        "training_rows": len(train_df),
        "n_features": meta["n_features_in"],
        "min_raw_pred": round(float(preds["raw_prediction"].min()), 1),
        "max_raw_pred": round(float(preds["raw_prediction"].max()), 1),
        "clipped_negatives": int(preds["was_clipped"].sum()),
    })

display(pd.DataFrame(summary_rows))
print("Ridge StandardScaler n_samples_seen_:", fitted["ridge"].named_steps["scaler"].n_samples_seen_)


## 3. Illustrative In-Sample Training-Fold Predictions (Diagnostics Only)
**Caution:** The rows below are **in-sample training-fold diagnostics** from `2024-11` (`target_period <= 2024-11`) solely to confirm finite, plausible prediction outputs. They do **not** measure out-of-sample forecasting performance. Validation evaluation occurs in Step 9.

In [ ]:
sample_diag = diag_df.loc[diag_df["target_period"] == pd.Timestamp("2024-11-01")].head(6)
display(sample_diag)
